# Stage 11 — DeepScoresV2 Dense Held-out Evaluation (mobile-resilient)

Run all cells once. After Drive authorization, evaluation runs as one long Colab task.
GPU is used when available; CPU is accepted for evaluation. Progress is written to Drive and reruns resume.
This improves interruption resilience but does not bypass or guarantee Google Colab runtime/session limits.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from contextlib import nullcontext
import hashlib,json,tarfile,random,math,io,time,os
import torch,torch.nn as nn,torch.nn.functional as F,numpy as np
from PIL import Image,ImageFilter
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

A=Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_DATA/ds2_dense.tar.gz")
O=Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_OUTPUT/deepscoresv2_dense_residual_unet_v1")
BEST=O/"best.pt"; X=Path("/content/st_score_restore_deepscoresv2_dense")
PROGRESS=O/"heldout_eval_progress.v1.json"; EVIDENCE=O/"heldout_final_evidence.json"
MD5="7237318e381e6e0848ec30eb82decb83"; SIZE=741814529
EXPECTED_BEST_SHA256="08b279161a9e8c4bd37376da221ecb4e07130724254ccf7d9591c8d32f368683"
EXPECTED_CONFIG="deff0f1270009839e234608dd9967038e1228a6b2b034e26059ac2f8cbfd0f80"
OFFICIAL_HELD_OUT_IMAGES=352; HELD_OUT_VARIANTS=2; SEED=20260907; PATCH=512; SAVE_EVERY=8
O.mkdir(parents=True,exist_ok=True)

def atomic_json(path,payload):
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(payload,indent=2,sort_keys=True))
    os.replace(tmp,path)

def digest(path,algo):
    h=hashlib.new(algo)
    with path.open("rb") as f:
        while b:=f.read(8<<20): h.update(b)
    return h.hexdigest()

if not A.exists():
    q=list(Path("/content/drive/MyDrive").rglob("ds2_dense.tar.gz"))
    if len(q)!=1: raise FileNotFoundError(f"Expected one ds2_dense.tar.gz, found {len(q)}")
    A=q[0]
if not BEST.exists(): raise FileNotFoundError(BEST)
if A.stat().st_size!=SIZE or digest(A,"md5")!=MD5: raise RuntimeError("Archive identity mismatch")
if digest(BEST,"sha256")!=EXPECTED_BEST_SHA256: raise RuntimeError("best.pt SHA256 mismatch")

def safe_extract():
    X.mkdir(parents=True,exist_ok=True); base=X.resolve()
    with tarfile.open(A,"r:gz") as t:
        for z in t.getmembers():
            p=z.name.replace(chr(92),"/")
            if not p or p.startswith("/") or ".." in Path(p).parts or z.issym() or z.islnk() or z.isdev():
                raise RuntimeError(f"unsafe tar member: {z.name}")
            q=(X/p).resolve()
            if q!=base and base not in q.parents: raise RuntimeError(f"tar escape: {z.name}")
        t.extractall(X)
if not X.exists() or not any(X.iterdir()): safe_extract()

te=list(X.rglob("deepscores_test.json"))
if len(te)!=1: raise RuntimeError("Expected one deepscores_test.json")
R=te[0].parent; raw=json.loads(te[0].read_text())
H=[str(r.get("file_name") or r.get("filename") or r.get("img_name")) for r in raw["images"] if (r.get("file_name") or r.get("filename") or r.get("img_name"))]
if len(H)!=OFFICIAL_HELD_OUT_IMAGES: raise RuntimeError(f"Expected 352 held-out images, got {len(H)}")

def ipath(n):
    for p in (R/"images"/n,R/n):
        if p.exists(): return p
    q=list(R.rglob(Path(n).name))
    if len(q)==1:return q[0]
    raise FileNotFoundError(n)

def rng(n,v):
    return random.Random(int.from_bytes(hashlib.sha256(f"{SEED}:heldout:{n}:{v}".encode()).digest()[:8],"big"))

def crop(im,r):
    w,h=im.size
    if min(w,h)<PATCH:
        s=max(PATCH/w,PATCH/h); im=im.resize((math.ceil(w*s),math.ceil(h*s)),Image.Resampling.BICUBIC); w,h=im.size
    a=np.asarray(im); best=(0,0); ink=-1
    for _ in range(8):
        x=r.randint(0,max(0,w-PATCH)); y=r.randint(0,max(0,h-PATCH)); q=(a[y:y+PATCH,x:x+PATCH]<235).mean()
        if q>ink: ink=q; best=(x,y)
    x,y=best; return im.crop((x,y,x+PATCH,y+PATCH))

def degrade(im,r):
    im=TF.rotate(im,r.uniform(-2.5,2.5),interpolation=InterpolationMode.BILINEAR,fill=255); w,h=im.size
    d=r.uniform(.002,.02); dx,dy=int(w*d),int(h*d)
    s=[[0,0],[w-1,0],[w-1,h-1],[0,h-1]]
    e=[[r.randint(0,dx),r.randint(0,dy)],[w-1-r.randint(0,dx),r.randint(0,dy)],[w-1-r.randint(0,dx),h-1-r.randint(0,dy)],[r.randint(0,dx),h-1-r.randint(0,dy)]]
    im=TF.perspective(im,s,e,interpolation=InterpolationMode.BILINEAR,fill=255)
    im=TF.adjust_brightness(im,r.uniform(.78,1.2)); im=TF.adjust_contrast(im,r.uniform(.82,1.18)); im=im.filter(ImageFilter.GaussianBlur(r.uniform(.15,1.75)))
    a=np.asarray(im).astype(np.float32)/255.; yy,xx=np.mgrid[:a.shape[0],:a.shape[1]]
    cx,cy=r.uniform(0,a.shape[1]),r.uniform(0,a.shape[0]); z=np.sqrt((xx-cx)**2+(yy-cy)**2); z/=max(z.max(),1)
    a*=1-r.uniform(0,.22)*(1-z); a+=np.random.default_rng(r.randrange(2**32)).normal(0,r.uniform(.002,.03),a.shape); a=np.clip(a,0,1)
    b=io.BytesIO(); Image.fromarray((a*255).astype("uint8")).save(b,"JPEG",quality=r.randint(52,95)); b.seek(0)
    return Image.open(b).convert("L")
def ten(im): return torch.from_numpy(np.asarray(im,dtype=np.float32)/255.).unsqueeze(0).unsqueeze(0)

class C(nn.Module):
    def __init__(self,a,b): super().__init__(); self.n=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.GroupNorm(8,b),nn.SiLU(),nn.Conv2d(b,b,3,padding=1),nn.GroupNorm(8,b),nn.SiLU())
    def forward(self,x): return self.n(x)
class UNet(nn.Module):
    def __init__(self,b=32):
        super().__init__(); self.e1=C(1,b);self.e2=C(b,2*b);self.e3=C(2*b,4*b);self.mid=C(4*b,8*b);self.d3=C(12*b,4*b);self.d2=C(6*b,2*b);self.d1=C(3*b,b);self.o=nn.Conv2d(b,1,1)
    def forward(self,x):
        a=self.e1(x);b=self.e2(F.max_pool2d(a,2));c=self.e3(F.max_pool2d(b,2));d=self.mid(F.max_pool2d(c,2))
        d=self.d3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode="bilinear",align_corners=False),c],1))
        d=self.d2(torch.cat([F.interpolate(d,size=b.shape[-2:],mode="bilinear",align_corners=False),b],1))
        d=self.d1(torch.cat([F.interpolate(d,size=a.shape[-2:],mode="bilinear",align_corners=False),a],1))
        return torch.clamp(x+torch.tanh(self.o(d))*.5,0,1)

D=torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE=torch.cuda.get_device_name(0) if D.type=="cuda" else "CPU"
m=UNet().to(D); ck=torch.load(BEST,map_location=D)
if ck.get("md5")!=MD5 or ck.get("cfg")!=EXPECTED_CONFIG or ck.get("epoch")!=19: raise RuntimeError("Checkpoint identity mismatch")
m.load_state_dict(ck["model"]); m.eval()
sx=torch.tensor([[-1.,0,1],[-2,0,2],[-1,0,1]],device=D).view(1,1,3,3); sy=sx.transpose(2,3)
def edge(x): return torch.sqrt(F.conv2d(x,sx,padding=1)**2+F.conv2d(x,sy,padding=1)**2+1e-6)
def state_sha():
    h=hashlib.sha256()
    for n,t in sorted(m.state_dict().items()): h.update(n.encode()); h.update(t.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
WEIGHTS_BEFORE=state_sha()

pairs=[(n,v) for n in H for v in range(HELD_OUT_VARIANTS)]
ORDER=hashlib.sha256(chr(10).join(f"{n}|{v}" for n,v in pairs).encode()).hexdigest(); TOTAL=len(pairs)
base=np.zeros(4); rest=np.zeros(4); start=0; resumed=False
if PROGRESS.exists():
    p=json.loads(PROGRESS.read_text())
    checks={"datasetId":"deepscoresv2.dense.v2","archiveMd5":MD5,"checkpointSha256":EXPECTED_BEST_SHA256,"orderSha256":ORDER,"totalPairs":TOTAL,"weightsBeforeSha256":WEIGHTS_BEFORE}
    for k,v in checks.items():
        if p.get(k)!=v: raise RuntimeError(f"Progress identity mismatch: {k}")
    start=int(p["nextIndex"]); base=np.array(p["baselineSum"],dtype=np.float64); rest=np.array(p["restoredSum"],dtype=np.float64); resumed=start>0 and start<TOTAL

def comp(a,b):
    px=F.l1_loss(a,b).item(); ed=F.l1_loss(edge(a),edge(b)).item(); mse=F.mse_loss(a,b).item()
    return np.array([px+.3*ed,px,ed,mse],dtype=np.float64)

print(f"Device={DEVICE}; resume={start}/{TOTAL}. Browser may go background; if runtime is terminated, Run all resumes from Drive progress.")
for i in range(start,TOTAL):
    n,v=pairs[i]; r=rng(n,v); y0=crop(Image.open(ipath(n)).convert("L"),r); x0=degrade(y0,r)
    x=ten(x0).to(D); y=ten(y0).to(D)
    with torch.inference_mode():
        ctx=torch.autocast("cuda",dtype=torch.float16) if D.type=="cuda" else nullcontext()
        with ctx: pred=m(x)
    base+=comp(x,y); rest+=comp(pred,y)
    if (i+1)%SAVE_EVERY==0 or i+1==TOTAL:
        atomic_json(PROGRESS,{"schemaVersion":"1.0.0","datasetId":"deepscoresv2.dense.v2","archiveMd5":MD5,"checkpointSha256":EXPECTED_BEST_SHA256,"checkpointConfigSha256":EXPECTED_CONFIG,"orderSha256":ORDER,"officialHeldOutImages":OFFICIAL_HELD_OUT_IMAGES,"heldOutVariants":HELD_OUT_VARIANTS,"totalPairs":TOTAL,"nextIndex":i+1,"baselineSum":base.tolist(),"restoredSum":rest.tolist(),"weightsBeforeSha256":WEIGHTS_BEFORE,"deviceLastSeen":DEVICE,"optimizerCreated":False,"backpropagationExecuted":False,"heldOutUsedForTraining":False,"heldOutUsedForTuning":False,"updatedAtUnix":int(time.time())})
        print(f"progress {i+1}/{TOTAL} ({100*(i+1)/TOTAL:.1f}%)")

def metrics(s):
    loss,pixel,ed,mse=(s/TOTAL).tolist()
    return {"loss":loss,"pixelL1":pixel,"edgeLoss":ed,"mse":mse,"psnrDb":10*math.log10(1/max(mse,1e-12))}
bm,rm=metrics(base),metrics(rest); WEIGHTS_AFTER=state_sha(); mutated=WEIGHTS_AFTER!=WEIGHTS_BEFORE
quality=(rm["loss"]<bm["loss"] and rm["pixelL1"]<bm["pixelL1"] and rm["edgeLoss"]<bm["edgeLoss"] and rm["psnrDb"]>bm["psnrDb"])
e={"artifactType":"stage11_deepscoresv2_dense_heldout_final_evidence","datasetId":"deepscoresv2.dense.v2","archiveMd5":MD5,"archiveChecksumVerified":True,"checkpointFile":"best.pt","checkpointSha256":EXPECTED_BEST_SHA256,"checkpointConfigSha256":EXPECTED_CONFIG,"checkpointEpoch":19,"executionDevice":DEVICE,"deviceType":D.type,"officialHeldOutImages":OFFICIAL_HELD_OUT_IMAGES,"heldOutVariants":HELD_OUT_VARIANTS,"evaluatedPairs":TOTAL,"resumedFromPersistedProgress":resumed,"progressFile":PROGRESS.name,"baseline":bm,"restored":rm,"qualityImproved":quality,"weightsBeforeSha256":WEIGHTS_BEFORE,"weightsAfterSha256":WEIGHTS_AFTER,"weightsMutated":mutated,"optimizerCreated":False,"backpropagationExecuted":False,"heldOutUsedForTraining":False,"heldOutUsedForTuning":False,"heldOutPass":bool(quality and not mutated),"finalStage11Pass":False,"stage9aPreservationEvaluationRequired":True,"productionInferenceAuthorized":False,"modelPublicationAuthorized":False,"stage12EntryAuthorized":False,"completedAtUnix":int(time.time())}
atomic_json(EVIDENCE,e); print(json.dumps(e,indent=2,sort_keys=True))


If the runtime is interrupted, reopen the notebook and choose **Run all**. The same archive/checkpoint/order identities are checked before resuming from `heldout_eval_progress.v1.json`. No `nohup`, keep-alive script, or idle-limit bypass is used.